<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building RAG Agents with LLMs</b></font></h1>
<h2><b>Notebook 2: </b>LLM 서비스와 AI Foundation Models</h2>
<br>

이 노트북에서는 LLM 서비스를 탐구합니다! 엣지 디바이스에 LLM을 배포하는 것의 장단점을 논의하고, NVIDIA AI Foundation Endpoints처럼 확장 가능한 서버 배포를 통해 강력한 모델을 최종 사용자에게 제공하는 방법을 살펴봅니다.

<br>

### **학습 목표:**

- LLM 서비스를 로컬에서 실행하는 것과 확장 가능한 클라우드 환경에서 실행하는 것의 장단점을 이해합니다.
- AI Foundation Model Endpoint 방식에 익숙해집니다. 여기에는 다음이 포함됩니다:
    - `curl`, `requests` 같은 패키지로 이루어지는 저수준의 원시 연결 인터페이스
    - 이 인터페이스가 LangChain 같은 오픈소스 소프트웨어와 매끄럽게 동작하도록 만들어진 추상화 계층
- 엔드포인트 풀에서 LLM 생성 결과를 받아오는 데 익숙해지고, 소프트웨어를 구축할 모델 부분집합을 선택할 수 있게 됩니다.

<br>

### **생각해 볼 질문:**

1. LLM 스택을 개발하는 사람에게는 어떤 종류의 모델 접근 권한을 주어야 하며, AI 기반 웹 애플리케이션의 최종 사용자에게 제공해야 하는 접근 권한과는 어떻게 다를까요?
2. 어떤 디바이스를 지원할지 고려할 때, 해당 디바이스의 로컬 컴퓨팅 리소스에 대해 어떤 경직된 가정을 하게 되며, 어떤 종류의 폴백(fallback)을 구현해야 할까요?
    - 고객에게 프라이빗 LLM 배포에 접근할 수 있는 Jupyter Labs 인터페이스를 제공하고 싶다면?
    - 이제 고객의 로컬 Jupyter Lab 환경에서 여러분의 프라이빗 LLM 배포를 지원하고 싶다면?
    - 임베디드 디바이스(예: Jetson Nano)를 지원하기로 한다면 무엇이 달라져야 할까요?
3. **[심화]** 클라우드 환경의 자체 컴퓨팅 인스턴스에 Stable Diffusion, Mixtral, Llama-13B를 배포해 같은 GPU 리소스를 공유하고 있다고 가정해 봅시다. 현재 Stable Diffusion에 대한 비즈니스 활용 사례는 없지만, 팀에서는 나머지 두 모델로 LLM 애플리케이션을 실험하고 있습니다. Stable Diffusion을 배포에서 제거해야 할까요?

<br>

----

<br>

## **Part 1**: 환경에 대규모 모델 가져오기

지난 노트북에서 현재 환경에는 할당된 클라우드 인스턴스 위에서 `docker-router`, `jupyter-notebook-server`, `frontend`, `llm_client` 등 여러 마이크로서비스가 실행 중이라는 점을 떠올려 보세요. 

- **jupyter-notebook-server**: 이 Jupyter Labs 세션을 실행하고 Python 환경을 호스팅하는 서비스. 
- **docker_router**: 최소한 마이크로서비스를 관찰하고 모니터링할 수 있게 도와주는 서비스.
- **frontend**: 간단한 채팅 인터페이스를 제공하는 라이브 웹사이트 마이크로서비스. 

이 노트북은 `llm_client` 마이크로서비스에 더 집중합니다. 여러분은 (적어도 내부적으로는) 이 서비스를 통해 여러 [**foundation model**](https://www.nvidia.com/en-us/ai-data-science/foundation-models/)과 상호작용하게 됩니다! 구체적으로는 [**NVIDIA AI Foundation Models**](https://catalog.ngc.nvidia.com/)의 일부를 사용해 AI 기반 파이프라인을 프로토타이핑하고, 자연어에 기반한 복잡한 애플리케이션을 오케스트레이션하게 됩니다.

$$---$$


거의 모든 도메인에서 거대한 딥러닝 모델을 배포하는 일은 흔하지만 어려운 작업입니다. Llama 2(70B 파라미터)나 Mixtral 7x8B 같은 앙상블 모델 등 오늘날의 모델은 고급 학습 기법, 방대한 데이터 리소스, 강력한 컴퓨팅 시스템의 산물입니다. 다행히 이 모델들은 이미 학습이 완료되어 있으며, 많은 활용 사례는 기성 솔루션으로도 이미 달성할 수 있습니다. 하지만 진짜 난관은 이 모델들을 효과적으로 호스팅하는 데 있습니다.

**대규모 모델의 배포 시나리오:**

1. **고급 데이터센터 배포:**
> NVIDIA [A100](https://www.nvidia.com/en-us/data-center/a100/)/[H100](https://www.nvidia.com/en-us/data-center/h100/)/[H200](https://www.nvidia.com/en-us/data-center/h200/) 같은 GPU를 갖춘 데이터센터 스택에서 압축·양자화되지 않은 모델을 실행하여 빠른 추론과 실험을 지원합니다.
> - **장점**: 확장 가능한 배포와 실험에 이상적이며, 대규모 학습 워크플로나 여러 사용자·모델을 동시에 지원하는 데 적합합니다.  
> - **단점:** 모델 학습/파인튜닝이나 저수준 모델 구성 요소를 다루는 경우가 아니라면, 서비스의 사용자마다 이 리소스를 할당하는 것은 비효율적입니다.

2. **중간 규모 데이터센터/특수 소비자용 하드웨어 배포:**
> 양자화 및 추가 최적화된 모델은 [L40](https://www.nvidia.com/en-us/data-center/l40/)/[A30](https://www.nvidia.com/en-us/data-center/products/a30-gpu/)/[A10](https://www.nvidia.com/en-us/data-center/products/a10-gpu/) 같은 보다 보수적인 데이터센터 GPU나, 심지어 VRAM이 큰 [RTX 40 시리즈 GPU](https://www.nvidia.com/en-us/geforce/graphics-cards/40-series/) 같은 일부 최신 소비자용 GPU에서도 (인스턴스당 한두 개씩) 실행할 수 있습니다.
> - **장점:** 단일 사용자 애플리케이션에서 추론 속도와 감당 가능한 제약 사이의 균형을 맞춘 구성입니다. 이런 세션은 사용자별로 배포하여 (양자화가 필요하더라도) 모델 내부에 직접 접근하면서 한두 개의 대규모 모델을 실행할 수도 있습니다.
> - **단점:** 사용자마다 인스턴스를 배포하는 것은 규모가 커지면 여전히 비용이 크지만, 일부 특수한 워크로드에서는 정당화될 수 있습니다. 반대로 사용자가 로컬 환경에서 이런 리소스를 갖추고 있으리라 가정하는 것은 비합리적일 가능성이 큽니다.

3. **소비자용 하드웨어 배포:**
> 신경망에 데이터를 통과시키는 능력은 크게 제한되지만, 대부분의 소비자용 하드웨어에는 그래픽 사용자 인터페이스(GUI), 인터넷에 연결된 웹 브라우저, 어느 정도의 메모리(최소 1GB는 안전하게 가정 가능), 그리고 꽤 강력한 CPU가 있습니다.
> - **단점:** 현재 대부분의 하드웨어는 어떤 구성으로도 로컬 대규모 모델을 한 번에 둘 이상 실행할 수 없으며, 하나만 실행하더라도 상당한 리소스 관리와 최적화 제약이 필요합니다.
> - **장점:** 서비스가 어떤 사용자를 지원해야 할지 고려할 때 합리적이고 포용적인 출발 가정이 됩니다.

이 코스에서 여러분의 환경은 전형적인 소비자용 하드웨어를 꽤 잘 대표합니다. 마이크로서비스로 시작하고 프로토타이핑할 수는 있지만, LLM 모델을 실행하기 어려운 CPU 전용 컴퓨팅 환경이라는 제약이 있습니다. 이는 큰 제약이지만, 다음을 통해 여전히 빠른 LLM 기능을 활용할 수 있습니다:
- 대규모 모델을 호스팅하는 컴퓨팅 가능 서비스에 대한 접근.
- 명령 입력과 결과 조회를 위한 간결한 인터페이스.

마이크로서비스와 포트 기반 연결에 대한 기초를 갖추었으니, 개발 환경에서 LLM에 접근하기 위한 효과적인 인터페이스 옵션을 탐색할 준비가 되었습니다!

----

<br>

## **Part 2:** 호스팅된 대규모 모델 서비스

CPU 전용 인스턴스처럼 리소스가 제한된 환경에서 대규모 언어 모델(LLM)에 접근하기 위해, 다양한 호스팅 옵션을 평가해 보겠습니다:

**블랙박스 호스팅 모델:**
> [**OpenAI**](https://openai.com/) 같은 서비스는 GPT-4 같은 블랙박스 모델과 상호작용할 수 있는 API를 제공합니다. 이런 강력하고 잘 통합된 서비스는 메모리를 자동으로 추적하고, 추가 모델을 호출하고, 필요에 따라 멀티모달 인터페이스를 통합하는 복잡한 파이프라인을 단순한 인터페이스로 제공하여 일반적인 사용 시나리오를 간소화합니다. 동시에 운영 방식이 불투명하고, 셀프 호스팅으로 가는 명확한 경로가 없는 경우가 많습니다.
> - **장점:** 바로 사용할 수 있고 일반 사용자의 진입 장벽이 낮습니다.
> - **단점:** 블랙박스 배포는 잠재적 프라이버시 문제, 제한된 커스터마이징, 규모가 커질 때의 비용 문제를 안고 있습니다.

**셀프 호스팅 모델:**

> 거의 모든 대규모 모델 배포의 뒤에는 확장 가능한 리소스와 초고속 대역폭을 갖춘 데이터센터에서 실행되는 하나 이상의 거대 모델이 있습니다. 대규모 모델을 규모 있게 배포하고 제공 인터페이스를 강하게 통제하려면 필수적이지만, 이런 시스템은 구축에 전문 지식이 필요하며 한 번에 한 사람의 비개발자 워크플로만 지원하기에는 대체로 적합하지 않습니다. 이런 시스템은 많은 동시 사용자, 여러 모델, 커스텀 인터페이스를 지원하는 데 훨씬 적합합니다.
> - **장점:** 커스텀 데이터셋과 API를 통합할 수 있으며, 주로 다수의 사용자를 동시에 지원하도록 설계되어 있습니다.
> - **단점:** 구축과 올바른 설정에 기술적 전문 지식이 필요합니다.

두 세계의 장점을 모두 취하기 위해 [**NVIDIA NGC Service**](https://www.nvidia.com/en-us/gpu-cloud/)를 활용합니다. NGC는 AI 솔루션을 설계하고 배포하기 위한 개발자 도구 모음을 제공합니다. 우리에게 핵심적인 것은 [NVIDIA AI Foundation Models](https://www.nvidia.com/en-us/ai-data-science/foundation-models/)로, 손쉬운 확장 배포(그대로 또는 추가 커스터마이징을 거쳐)를 위해 미리 튜닝되고 최적화된 모델들입니다. 또한 NGC는 [확장 가능한 DGX 가속 컴퓨팅 환경](https://www.nvidia.com/en-us/data-center/dgx-platform/)에서 라이브 foundation model을 조회할 수 있는 접근 가능한 모델 엔드포인트를 호스팅합니다.

----

<br>

## **Part 3:** 호스팅된 추론 시작하기

**대규모 추론을 위해 모델을 배포할 때 일반적으로 거쳐야 하는 단계는 다음과 같습니다:**
- 사용자가 접근할 모델을 정하고, 이를 호스팅할 리소스를 할당합니다.
- 사용자에게 어떤 종류의 제어권을 줄지 정하고, 이에 접근할 방법을 노출합니다.
- 사용량을 추적/제한하는 모니터링 체계를 만들고, 필요에 따라 확장하고 스로틀링하는 시스템을 구축합니다.

이 코스에서는 OpenAI 호환 API를 노출하는 NVIDIA 호스팅 [**NVIDIA NIM Microservices**](https://www.nvidia.com/en-us/ai-data-science/products/nim-microservices/)를 사용합니다. 배포 토폴로지, 용량, 배칭, 라우팅, 스케일링은 서비스가 담당하며, 이런 세부 사항은 달라질 수 있습니다. 우리의 클라이언트는 특정 백엔드 구성이 아니라 안정적인 API 경계에 의존합니다.

아래 이미지는 커스텀(OpenAI 호환이 아닌) API를 통한 임의의 함수 호출을 보여 줍니다. 배포 세부 사항은 서비스 쪽에 맡겨 두더라도 서비스 경계를 이해하는 데 유용합니다. 

<!-- > <img style="max-width: 1000px;" src="imgs/ai-playground-api.png" /> -->
<!-- > <img src="https://drive.google.com/uc?export=view&id=1ckAIZoy7tvtK1uNqzA9eV5RlKMbVqs1-" width=1000px/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/ai-playground-api.png" width=800px/>

**게이트웨이 측:** 이 API를 더 표준적으로 만들기 위해, API 게이트웨이 서버가 이런 함수들을 OpenAI 호환 chat-completions API 및 스키마 뒤에 모아 둡니다. 이 호환성 덕분에 OpenAI 클라이언트를 유효한 인터페이스로 사용할 수 있습니다.

이 코스에서는 LangChain이라는 LLM 오케스트레이션 프레임워크(자세한 내용은 뒤에서)에 연결되는 더 특화된 인터페이스를 사용하게 됩니다. 여러분 쪽에서는 [`langchain_nvidia_ai_endpoints`](https://python.langchain.com/docs/integrations/chat/nvidia_ai_endpoints/) 라이브러리의 `ChatNVIDIA` 같은 맞춤형 인터페이스를 사용하게 됩니다. *자세한 내용은 뒤에서 다룹니다.*

**사용자 측:** 이 엔드포인트들을 클라이언트에 통합하면, 생성형 AI 기능을 활용해 애플리케이션에 추론과 생성 능력을 부여하는 통합, 파이프라인, 사용자 경험을 설계할 수 있습니다. 이런 애플리케이션의 대표적인 예가 [**OpenAI의 ChatGPT**](https://chat.openai.com/)로, GPT4, Dalle 등 여러 엔드포인트를 오케스트레이션한 것입니다. 때로는 하나의 지능적인 모델처럼 보이지만, 실제로는 상태와 컨텍스트 제어를 관리하는 소프트웨어 엔지니어링이 결합된 모델 엔드포인트의 집합일 뿐입니다. 이 점은 코스 전반에 걸쳐 반복해서 강조될 것이며, 코스가 끝날 무렵에는 임의의 활용 사례를 위해 비슷한 채팅 어시스턴트를 어떻게 만들 수 있을지 감을 잡게 될 것입니다. 

<!-- > <img style="max-width: 700px;" src="imgs/openai_chat.png" /> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/openai_chat.png" width=700px/>

----

<br>

## **Part 4: [실습]** Foundation Model Endpoint 사용해 보기

이 섹션에서 드디어 LLM 엔드포인트와 상호작용해 봅니다! 

**여러분의 개인 환경에서**: [`build.nvidia.com`](https://build.nvidia.com/)에 접속하여 사용하고 싶은 모델을 찾으세요. 예를 들어 [**NVIDIA Nemotron 3.5 Lightning 모델**](https://build.nvidia.com/nvidia/nemotron-3.5-lightning-30b-a3b) 페이지에는 사용 예시, 추가 자료, 그리고 "Apply To Self-Host"와 "Get API Key" 같은 옵션이 있습니다.

- **"Apply To Self-Host"** 를 클릭하면 NVIDIA Microservices에 대한 정보로 안내되며, 가입(얼리 액세스/NVIDIA AI Enterprise 경로)하거나 알림 목록에 등록(General Access 경로)할 수 있는 방법을 제공합니다.

- **"Get API Key"** 를 클릭하면 "nvapi-"로 시작하는 API 키가 생성되며, 이를 네트워크 요청을 통해 API 엔드포인트에 제공할 수 있습니다!

이렇게 하려면 다음과 같이 API 키를 노트북에 추가해야 합니다:

In [ ]:
# import os
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."

<br/>

**코스 환경에서**: 요청은 [**`./composer/microservices/llm_client.py`**](./composer/microservices/llm_client.py)에 정의된 서버를 사용합니다. 노트북 클라이언트는 하나의 base URL을 유지하고, 서버는 요청을 NVIDIA API로 전달하거나 임베딩 및 리랭킹용 상주 CPU 모델을 실행합니다. 따라서 뒤편의 배포가 바뀌더라도 클라이언트 인터페이스는 그대로 유지될 수 있습니다.

기본 모델 목록에는 코스에서 사용하는 채팅 모델만 포함되어 있습니다. 별도의 `scope=all` 쿼리를 사용하면 더 넓은 라이브 카탈로그와 로컬 검색 모델을 탐색할 수 있습니다. API 자격 증명은 노트북이 아니라 서버 환경에 보관됩니다.

<br/>

### **4.1.** 수동 Python 요청

앞서 말했듯이 Python의 `requests` 라이브러리로 마이크로서비스나 원격 API와 상호작용할 수 있으며, 일반적으로 다음 과정을 따릅니다:
- **라이브러리 임포트:** HTTP 요청을 위한 requests와 JSON 데이터 처리를 위한 json을 임포트합니다.
- **API URL과 헤더:** API 엔드포인트의 URL과 헤더(인증(API 키), 데이터 형식 설정 포함)를 정의합니다.
- **데이터 페이로드:** 보내고자 하는 데이터를 지정합니다. 여기서는 간단한 쿼리입니다.
- **요청 보내기:** `requests.post`로 POST 요청을 보냅니다. API 요구 사항에 따라 post 대신 `get`, `put` 등을 사용할 수 있습니다.
- **응답 처리:** 상태 코드로 요청 성공 여부를 확인(200이면 성공)한 뒤 데이터를 처리합니다.

먼저 서비스 상태를 확인한 다음, 코스용 카탈로그와 더 넓은 탐색용 카탈로그를 비교해 보겠습니다:

In [ ]:
import requests

service_url = "http://llm_client:9000"
headers = {"content-type": "application/json"}

service = requests.get(f"{service_url}/health", headers=headers, timeout=10).json()
service

In [ ]:
models_url = f"{service_url}/v1/models"
course_models = requests.get(models_url, headers=headers, timeout=10).json()["data"]
all_models = requests.get(
    models_url, headers=headers, params={"scope": "all"}, timeout=10
).json()["data"]

print("Course chat models:")
for model in course_models:
    print(" -", model["id"])

local_models = [
    model for model in all_models if model.get("owned_by") == "course-runtime"
]
print(f"\nFull exploration catalog: {len(all_models)} models")
print("Local retrieval models:")
for model in local_models:
    print(f" - {model['model_type']}: {model['id']}")

# Evaluate all_models when you want to inspect the complete catalog.
course_models[0] if course_models else service

<br/>

이로써 상위 수준의 커넥터를 추가하기 전에 클라이언트/서버 경계를 확인했습니다. 뒤에서 사용할 채팅, 임베딩, 랭킹 클라이언트는 이 base URL 아래의 서로 다른 경로로 같은 종류의 요청을 보냅니다.

In [ ]:
from getpass import getpass
import os

## Where are you sending your requests?
invoke_url = "http://llm_client:9000/v1/chat/completions"

## If you wanted to use your own API Key, it's very similar
# if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
#     os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA_API_KEY: ")
# invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"

## If you wanted to use OpenAI, it's very similar
# if not os.environ.get("OPENAI_API_KEY", "").startswith("sk-"):
#     os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
# invoke_url = "https://api.openai.com/v1/chat/completions"

## Meta communication-level info about who you are, what you want, etc.
headers = {
    "accept": "text/event-stream",
    "content-type": "application/json",
    # "Authorization": f"Bearer {os.environ.get('NVIDIA_API_KEY')}",
    # "Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY')}",
}

## Arguments to your server function
payload = {
    "model": "nvidia/nemotron-3.5-lightning-30b-a3b",
    "messages": [{"role":"user","content":"Tell me hello in French"}],
    "temperature": 0.5,   
    "top_p": 1,
    "max_tokens": 1024,
    "stream": True,
    "chat_template_kwargs": {"enable_thinking": False},
}

In [ ]:
import requests
import json

## Use requests.post to send the header (streaming meta-info) the payload to the endpoint
## Make sure streaming is enabled, and expect the response to have an iter_lines response.
response = requests.post(invoke_url, headers=headers, json=payload, stream=True, timeout=(10, 300))

## If your response is an error message, this will raise an exception in Python
try: 
    response.raise_for_status()  ## If not 200 or similar, this will raise an exception
except Exception as e:
    # print(response.json())
    print(response.json())
    raise e

## Custom utility to make live a bit easier
def get_stream_token(entry: bytes):
    """Utility: Coerces out ['choices'][0]['delta'][content] from the bytestream"""
    if not entry: return ""
    entry = entry.decode('utf-8')
    if entry.startswith('data: '):
        try: entry = json.loads(entry[5:])
        except ValueError: return ""
    return (entry.get('choices') or [{}])[0].get('delta', {}).get('content') or ""

## If the post request is honored, you should be able to iterate over 
for line in response.iter_lines():
    
    ## Without Processing: data: {"id":"...", ... "choices":[{"index":0,"delta":{"role":"assistant","content":""}...}...
    # if line: print(line.decode("utf-8"))

    ## With Processing: An actual stream of tokens printed one-after-the-other as they come in
    print(get_stream_token(line), end="")

<br>

#### **[참고]**

**채팅 모델이 "messages"를 입력으로 기대한다는 점을 눈치채셨을 겁니다:**

로컬 HuggingFace 모델 같은 원시 LLM 인터페이스에 익숙하다면 낯설 수 있지만, OpenAI 모델 사용자에게는 꽤 표준적으로 보일 것입니다. 원시 텍스트 완성 대신 제한된 인터페이스를 강제함으로써 서비스는 사용자가 할 수 있는 일을 더 잘 통제할 수 있습니다. 이 인터페이스에는 많은 장단점이 있으며, 주목할 만한 것은 다음과 같습니다:
- 서비스가 특정 역할 유형이나 파라미터의 사용을 제한할 수 있습니다(예: 시스템 메시지 제한, 임의 생성을 유도하는 프라이밍 메시지 등).
- 서비스가 커스텀 프롬프트 형식을 강제하고, 채팅 인터페이스에 의존하는 추가 옵션을 내부적으로 구현할 수 있습니다.
- 서비스가 더 강한 가정을 바탕으로 추론 파이프라인에 더 깊은 최적화를 적용할 수 있습니다.
- 서비스가 기존 생태계와의 호환성을 활용하기 위해 다른 인기 있는 인터페이스를 모방할 수 있습니다.

이 모두가 타당한 이유이며, 서비스를 선택하거나 직접 배포할 때 특정 활용 사례에 어떤 인터페이스 옵션이 가장 적합한지 고려하는 것이 중요합니다.

**모델을 조회하는 두 가지 근본적인 방식이 있다는 점도 눈치채셨을 겁니다:**

**스트리밍 없이 호출(invoke)** 할 수 있습니다. 이 경우 서비스 응답은 전부 계산이 끝난 뒤 한 번에 도착합니다. 다른 작업을 하기 전에 모델의 전체 출력이 필요할 때 좋습니다. 예를 들어 전체 결과를 출력하거나 다운스트림 작업에 사용할 때입니다. 응답 본문은 대략 다음과 같습니다:

```json
{
    "id": "d34d436a-c28b-4451-aa9c-02eed2141ed3",
    "choices": [{
        "index": 0,
        "message": { "role": "assistant", "content": "Bonjour! ..." },
        "finish_reason": "stop"
    }],
    "usage": {
        "completion_tokens": 450,
        "prompt_tokens": 152,
        "total_tokens": 602
    }
}
```

**스트리밍으로 호출**할 수도 있습니다. 이 경우 하나의 요청에 대해 마지막 마커가 도착할 때까지 응답이 청크 단위로 전달됩니다. 응답이 도착하는 대로 사용할 수 있을 때 좋습니다(생성되는 즉시 사용자에게 출력을 보여 주는 언어 모델 구성 요소에 매우 적합합니다). 이 경우 응답 본문은 다음과 훨씬 비슷합니다:

```json
data:{"id":"...","choices":[{"index":0,"delta":{"role":"assistant","content":"Bon"},"finish_reason":null}]}
data:{"id":"...","choices":[{"index":0,"delta":{"role":"assistant","content":"j"},"finish_reason":null}]}
...
data:{"id":"...","choices":[{"index":0,"delta":{"role":"assistant","content":""},"finish_reason":"stop"}]}
data:[DONE]
```

두 옵션 모두 Python의 `requests` 라이브러리로 비교적 쉽게 구현할 수 있지만, 인터페이스를 그대로 사용하면 반복적인 코드가 많아집니다. 다행히 이를 훨씬 쉽게 사용하고 더 큰 프로젝트에 통합할 수 있게 해 주는 시스템들이 있습니다!

<br/>

### **4.2.** OpenAI 클라이언트 요청

이런 인터페이스가 존재한다는 것을 알아 두는 것은 좋지만, 그대로 사용하면 반복적인 코드와 불필요한 복잡성이 많아집니다. 다행히 이를 훨씬 쉽게 사용하고 더 큰 프로젝트에 통합할 수 있게 해 주는 시스템들이 있습니다! `requests`보다 한 단계 높은 추상화는 OpenAI 클라이언트처럼 더 정형화된 클라이언트를 사용하는 것입니다. NVIDIA 엔드포인트는 OpenAI 호환 API와 스키마를 노출하므로 OpenAI 클라이언트를 사용할 수 있습니다. 내부적으로는 여전히 같은 과정이 수행되며, 아마도 [**httpx**](https://github.com/encode/httpx)나 [**aiohttp**](https://github.com/aio-libs/aiohttp) 같은 저수준 클라이언트가 이를 담당합니다. 

In [ ]:
## Using General OpenAI Client
from openai import OpenAI

# client = OpenAI()  ## Assumes OPENAI_API_KEY is set

# client = OpenAI(
#     base_url = "https://integrate.api.nvidia.com/v1",
#     api_key = os.environ.get("NVIDIA_API_KEY", "")
# )

client = OpenAI(
    base_url = "http://llm_client:9000/v1",
    api_key = "I don't have one",
    timeout = 300,
)

completion = client.chat.completions.create(
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
    messages=[{"role":"user","content":"Hello World! Tell me about birds!"}],
    temperature=1,
    top_p=1,
    max_tokens=1024,
    stream=True,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

## Streaming with Generator: Results come out as they're generated
for chunk in completion:
    if not chunk.choices:
        continue
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")

In [ ]:
## Non-Streaming: Results come from server when they're all ready
completion = client.chat.completions.create(
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
    # model="gpt-4-turbo-2024-04-09",
    messages=[{"role":"user","content":"Hello World"}],
    temperature=1,
    top_p=1,
    max_tokens=1024,
    stream=False,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

completion

<br/>

### **4.3.** ChatNVIDIA 클라이언트 요청

지금까지 **원시 요청(raw requests)** 과 **API 클라이언트**라는 두 추상화 계층에서 통신이 이루어지는 것을 보았습니다. 이 코스에서는 LangChain이라는 프레임워크로 LLM 오케스트레이션을 하려고 하므로, 한 단계 더 높은 추상화인 **프레임워크 커넥터(Framework Connector)** 로 올라가야 합니다.

**커넥터**의 목표는 임의의 API를 원래 형태에서 대상 코드베이스가 기대하는 형태로 변환하는 것입니다. 이 코스에서는 LangChain의 활발한 체인 중심 생태계를 활용하고자 하는데, 원시 `requests` API만으로는 거기까지 갈 수 없습니다. 내부적으로 로컬에서 호스팅되지 않는 모든 LangChain 채팅 모델은 이런 API에 의존해야 하지만, 개발자에게 보이는 API는 기본 파라미터와 `invoke`, `stream` 같은 간단한 유틸리티 함수를 갖춘 훨씬 깔끔한 [`LLM` 또는 `SimpleChatModel` 스타일 인터페이스](https://reference.langchain.com/python/langchain-core/language_models/chat_models/BaseChatModel)입니다.

LangChain 인터페이스 탐색의 시작으로, `ChatNVIDIA` 커넥터를 사용해 `chat/completions` 엔드포인트와 상호작용해 보겠습니다. 이 모델은 LangChain 확장 생태계의 일부이며 `pip install langchain-nvidia-ai-endpoints`로 로컬에 설치할 수 있습니다.

In [ ]:
## Using ChatNVIDIA
from langchain_nvidia_ai_endpoints import ChatNVIDIA

## NVIDIA_API_KEY pulled from environment
llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})
# llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", mode="open", base_url="http://llm_client:9000/v1", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})
llm.invoke("Hello World")

In [ ]:
llm._client.last_inputs

In [ ]:
# llm._client.last_response
llm._client.last_response.json()

<br/>

#### **[참고]**

- **이 코스는 공개 `langchain-nvidia-ai-endpoints` 커넥터를 사용하며, 여기서 사용하는 OpenAI 호환 엔드포인트를 위해 `ChatNVIDIA`를 제공합니다.**

- **ChatNVIDIA가 기본적으로 `llm_client` 마이크로서비스를 사용하는 이유는 그렇게 되도록 환경 변수를 설정해 두었기 때문입니다**: 

In [ ]:
import os

{k:v for k,v in os.environ.items() if k.startswith("NVIDIA_")}
## Translation: Use the base_url of llm_client:9000 for the requests,
## and use "open"api-spec access for model discovery and url formats

<br/>

이 코스에서는 텍스트 생성에 `nvidia/nemotron-3.5-lightning-30b-a3b`를 사용합니다. 일반적인 채팅 및 지시(instruction) 호출에서는 thinking을 비활성화하지만, 다음 예시는 추론 출력을 확인할 수 있도록 명시적으로 활성화합니다. `meta/llama-3.2-11b-vision-instruct`는 이미지 입력이 필요한 인터페이스에서만 사용할 수 있습니다. 일상적인 모델 조회가 읽기 쉽도록 기본 목록은 이 역할들로만 제한되어 있습니다.

다른 모델을 시도하고 싶다면 4.1절의 전체 탐색 카탈로그를 사용하세요. 코스 예제에 대체 적용하기 전에 해당 모델의 엔드포인트와 응답 형식이 클라이언트와 맞는지 확인하세요.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

model_list = ["nvidia/nemotron-3.5-lightning-30b-a3b"]

for model_name in model_list:
    llm = ChatNVIDIA(
        model=model_name,
        timeout=300,
        temperature=1,
        top_p=0.95,
        max_completion_tokens=1536,
        model_kwargs={
            "chat_template_kwargs": {"enable_thinking": True},
            "reasoning_budget": 1024,
        },
    )
    print(f"TRIAL: {model_name}")
    try: 
        last = -1
        for chunk in llm.stream("Tell me about yourself! 2 sentences."):
            ## SOME CHUNKS MAY CARRY FINAL RESPONSES
            if chunk.content: 
                if last != 0: 
                    print("[ RESPONSE  ] ", end="")
                    last = 0
                print(chunk.content, end="")
            ## SOME CHUNKS MAY CARRY REASONING (PRE-RESPONSE) DATA
            elif chunk.additional_kwargs.get("reasoning_content"): 
                if last != 1:
                    print("[ REASONING ] ", end="")
                    last = 1
                print(chunk.additional_kwargs.get("reasoning_content"), end="")
            ## AND SOME CHUNKS MAY CARRY ADDITIONAL METADATA, LIKE USAGE/STATUS
            else: 
                if last != 2:
                    print()
                    last = 2
                # print(" - ", repr(chunk))
                print("[ METADATA  ]", {k:v for k,v in chunk.__dict__.items() if bool(v)})
    except Exception as e: 
        print(f"EXCEPTION: {e}")    ## If some models fail, feel free to use others
    except KeyboardInterrupt:
        print(f"Stopped manually")  ## Feel free to hit square while running
        break
    print("\n\n" + "="*84)

<a href="/jaeger" target="_blank" style="display: inline-block; padding: 10px 20px; background-color: #119999; color: white; text-decoration: none; border-radius: 4px; font-weight: bold;">Jaeger UI 열기</a>

----

<br>

## **Part 5:** 마무리

이 노트북의 목표는 LLM 서비스 호스팅 전략에 대한 논의를 제공하고 AI Foundation Model 엔드포인트를 소개하는 것이었습니다. 이 과정에서 원격 LLM 시스템을 어떻게 제공하고 엣지 디바이스에서 어떻게 접근할 수 있는지 직관적으로 이해하셨기를 바랍니다!

### <font color="#76b900">**수고하셨습니다!**</font>

### **다음 단계:**
1. **[선택]** 노트북 상단의 **"생각해 볼 질문" 섹션**을 다시 읽고 가능한 답을 생각해 보세요.

<br>

---

<div style="width: 55%%; background-color: white; margin-top: 50px;"><center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png" width="300" /></a></center></div>